# Modern Hopfield Network — remplace le WTA par une projection continue

Pour toute couche $l$, la projection ne se fait plus par argmax brutal mais par la **dynamique de rappel de Modern Hopfield** :
$$z = \text{softmax}(\beta \cdot W x)$$

- $\beta$ (inverse de température) : **levier de contrôle ultime**
  - $\beta \to \infty$ : retrouve un **WTA dur** (1 neurone à 100%)
  - $\beta$ modéré : **consensus continu et lissé** (élimine le bruit sans perdre la compétition)

Reconstruction : $x_{rec} = W^T z$ (combinaison convexe des motifs).
Surprise continue : $\mathcal{S}_{auto} = \|x - W^T \cdot \text{softmax}(\beta W x)\|^2$.
Plasticité Oja **pondérée par $z$ continu** (au lieu d'une activation binaire).

## 1. Effet de β : WTA dur vs consensus lissé

In [1]:
# Modern Hopfield Network (remplace le WTA)
import os, json
import numpy as np
import torch
import matplotlib.pyplot as plt
from recherche_agi import (project_hopfield, surprise, oja_hopfield_update,
                           HierarchicalCOCO)

rng = np.random.default_rng(0)
W = rng.random((5, 20)); W /= np.linalg.norm(W, axis=1, keepdims=True)
x = rng.random(20); x /= np.linalg.norm(x)

betas = [0.1, 1, 5, 50, 500]
print("β  |  distribution z (projection MHN)")
for b in betas:
    z = project_hopfield(x, W, beta=b)
    print(f"{b:>5g} | " + " ".join(f"{v:.2f}" for v in z))
print()
print("=> β faible : consensus lissé (toutes les activations contribuent)")
print("=> β élevé  : WTA dur (1 neurone dominant, comme l'ancien argmax)")

β  |  distribution z (projection MHN)
  0.1 | 0.20 0.20 0.20 0.20 0.20
    1 | 0.20 0.20 0.21 0.19 0.20
    5 | 0.21 0.21 0.26 0.14 0.18
   50 | 0.11 0.09 0.78 0.00 0.02
  500 | 0.00 0.00 1.00 0.00 0.00

=> β faible : consensus lissé (toutes les activations contribuent)
=> β élevé  : WTA dur (1 neurone dominant, comme l'ancien argmax)


## 2. Surprise continue + plasticité Oja pondérée

In [2]:
# surprise continue
S, z, x_rec = surprise(x, W, beta=5.0)
print(f"Surprise continue S_auto = {S:.4f}")
print(f"  z somme = {z.sum():.3f} (distribution)", f"| x_rec = W^T·z")

# plasticité Oja pondérée par z continu
W2, z2 = oja_hopfield_update(W, x, beta=5.0, lr=0.1)
print(f"Oja pondérée : max|ΔW| = {np.abs(W2-W).max():.5f}, z somme = {z2.sum():.3f}")
print("=> Les poids convergent vers les attracteurs (pas une activation binaire)")

Surprise continue S_auto = 0.2730
  z somme = 1.000 (distribution) | x_rec = W^T·z
Oja pondérée : max|ΔW| = 0.00701, z somme = 1.000
=> Les poids convergent vers les attracteurs (pas une activation binaire)


## 3. Intégration dans le pipeline hiérarchique

In [3]:
d = np.load('../data/coco_stuff/features_val.npz')
features = {int(k): v for k, v in d.items()}
d_in = next(iter(features.values())).shape[1]

model = HierarchicalCOCO(d_in=d_in, n_init=10, novelty_threshold=0.6, max_neurons=2000,
                         surprise_plateau=0.002, plateau_window=200, min_neurons_before_spawn=30,
                         beta=5.0)   # MHN actif
surp = []
for ci, cls in enumerate(sorted(features.keys())[:30]):
    X = features[cls][:60]
    gw = int(np.ceil(np.sqrt(len(X)))); gh = int(np.ceil(len(X)/gw))
    s = model.step_image(X, np.full(len(X), cls), gh, gw)
    surp.append(s)
s = model.summary()
print(f"Modèle : {s['layers']} couches, tailles {s['layer_sizes']}, {s['skips']['n_connections']} skips")
print(f"Surprise : {surp[0]:.3f} -> {surp[-1]:.3f} (descend, le MHN apprend)")

Modèle : 1 couches, tailles [14], 0 skips
Surprise : 1.559 -> 0.614 (descend, le MHN apprend)


## 4. Analyse

In [4]:
print("=== ANALYSE : MODERN HOPFIELD NETWORK ===")
print("1. Le WTA (argmax brutal) est remplacé par z = softmax(β·Wx).")
print("2. β contrôle le compromis : WTA dur (β grand) vs consensus lissé (β modéré).")
print("3. La surprise S_auto = ||x - W^T·z||² est continue et dérivable.")
print("4. La plasticité Oja est pondérée par z continu (converge vers attracteurs).")
print("5. Le reste de l'architecture (hiérarchie, skip, message passing) est inchangé.")

=== ANALYSE : MODERN HOPFIELD NETWORK ===
1. Le WTA (argmax brutal) est remplacé par z = softmax(β·Wx).
2. β contrôle le compromis : WTA dur (β grand) vs consensus lissé (β modéré).
3. La surprise S_auto = ||x - W^T·z||² est continue et dérivable.
4. La plasticité Oja est pondérée par z continu (converge vers attracteurs).
5. Le reste de l'architecture (hiérarchie, skip, message passing) est inchangé.
